<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/vision/segmentation-metrics/segmentation-metrics.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("segmentation-metrics", ...)`


# Segmentation Metrics: Confusion Matrix, mIoU and Dice

**Difficulty**: 🟢 Easy

### Problem Statement

Semantic segmentation models (FCN, DeepLab, SegFormer, Mask2Former, and the segmentation heads of vision foundation models) predict a class for every pixel. Nearly every number reported for them — mIoU on Cityscapes, ADE20K and PASCAL VOC, Dice on medical benchmarks, pixel accuracy — is computed from one $C \times C$ **confusion matrix of pixel counts**. The formulas are easy. What goes wrong are the conventions, which change the headline number without raising any error: which pixels are ignored (the 255 "void" label), what an absent class contributes, and whether you average over images or pool over the dataset.

Build the evaluation toolkit: vectorized, streaming, and exact on those conventions. This problem is about **evaluation**. The differentiable soft-Dice *loss* is in **losses-from-scratch**, and confusion-matrix-based precision/recall for classification is in **classification-metrics**.

**Confusion matrix.** `cm[i, j]` is the number of counted pixels whose ground truth is class $i$ and whose prediction is class $j$. **Rows are ground truth** and columns are predictions, the same as sklearn, torchmetrics and FCN's $n_{ij}$. For class $c$:
$$TP_c = cm_{cc}, \qquad FN_c = \sum_j cm_{cj} - TP_c \ \ (\text{row}), \qquad FP_c = \sum_i cm_{ic} - TP_c \ \ (\text{column}).$$

**Metrics.**
$$\mathrm{IoU}_c = \frac{TP_c}{TP_c + FP_c + FN_c}, \qquad \mathrm{Dice}_c = \frac{2\,TP_c}{2\,TP_c + FP_c + FN_c} = \frac{2\,\mathrm{IoU}_c}{1 + \mathrm{IoU}_c},$$
$$\text{pixel accuracy} = \frac{\sum_c TP_c}{\sum_{i,j} cm_{ij}}, \qquad \text{mean class accuracy} = \underset{c\,:\,\text{row}_c > 0}{\operatorname{mean}} \ \frac{TP_c}{\sum_j cm_{cj}}, \qquad \mathrm{mIoU} = \underset{c\,:\,TP_c + FP_c + FN_c > 0}{\operatorname{mean}} \ \mathrm{IoU}_c .$$
These are FCN's pixel accuracy, mean accuracy and mean IU (Long et al., 2015, Sec. 5). Dice per class is the F1 score of that class.

**Conventions (graded).**

- **Which pixels count.** A pixel is counted iff its *target* is a class id in $[0, C)$ and is not `ignore_index`. Targets outside $[0, C)$ are void. `ignore_index` only drops pixels by their **target** value. When it is a valid class id, e.g. `ignore_index=0`, that class has no ground-truth pixels left, but a prediction of it on a counted pixel is still an error that lands in its column, so its IoU is 0 if it is ever predicted and NaN otherwise. MMSegmentation does the same; torchmetrics' macro mIoU additionally leaves that class out of the average.
- **Absent classes.** A class with $TP + FP + FN = 0$ (not in the target, never predicted) has **NaN** IoU and Dice, and the means skip it (`torch.nanmean`). A class that is only in the target, or only predicted, scores **0**, not NaN.
- **Dataset-level pooling.** Over a dataset you **sum the confusion matrices of all images and compute the metrics once**. This is how Cityscapes (TP, FP and FN "determined over the whole test set"), PASCAL VOC, ADE20K and MMSegmentation report mIoU. It is **not** the mean of per-image mIoUs. Averaging per image gives every image the same weight whatever its size, and treats a 3-pixel sliver of a class that the model missed in one image (IoU 0) as seriously as a large, well-segmented region in another. Pooling weighs every pixel equally. On the toy dataset below the two differ by 24 points.

---

### Requirements

1. **`confusion_matrix(pred, target, num_classes, ignore_index=255) -> LongTensor`** of shape `(C, C)`, int64.
   - `pred` and `target` are integer label maps of the **same, arbitrary shape** (`(H, W)`, `(B, H, W)`, flattened, ...) and any integer dtype, including the `uint8` you get from PNG masks.
   - Count with **one `torch.bincount`** (or another loop-free counting op). No Python loop over pixels or classes.
   - Drop pixels whose target is outside $[0, C)$ or equals `ignore_index`. `ignore_index=None` means no value is ignored, though the range test still applies.
   - If a **counted** pixel has a prediction outside $[0, C)$, raise `ValueError`. Never look at predictions on dropped pixels.
   - Do not modify the inputs. If nothing is counted, return a zero matrix.
2. **`iou_per_class(cm) -> Tensor`**: `(C,)` float64, NaN for absent classes.
3. **`mean_iou(cm) -> Tensor`**: 0-dim float64, the mean over non-NaN IoUs (NaN if every class is absent).
4. **`dice_per_class(cm) -> Tensor`**: `(C,)` float64, NaN for absent classes.
5. **`pixel_accuracy(cm) -> Tensor`**: 0-dim float64 (NaN for an empty matrix).
6. **`mean_class_accuracy(cm) -> Tensor`**: 0-dim float64, the mean of $TP_c / \text{row}_c$ over the classes that have at least one ground-truth pixel. A class that is predicted but never in the target is skipped.
7. **`SegMetric(num_classes, ignore_index=255)`**, a streaming accumulator:
   - `update(pred, target)` adds that batch's confusion matrix to an **int64** `(C, C)` buffer. Memory stays $O(C^2)$: do not store predictions.
   - `compute()` returns a dict with keys `"confusion_matrix"` (an int64 copy of the summed matrix), `"iou"`, `"mean_iou"`, `"dice"`, `"mean_dice"` (nanmean of Dice), `"pixel_accuracy"` and `"mean_class_accuracy"`, all computed on the **summed** matrix. Calling it does not change the state.
   - `reset()` clears the buffer.

Scalars may be 0-dim tensors or Python floats.

---

### Constraints

- ✅ `torch.bincount`, `reshape`, `diagonal`, `sum`, boolean masks, `torch.nanmean`.
- ❌ No sklearn or torchmetrics (the optional cross-check cell may import sklearn).
- ❌ No Python loop over pixels or classes in `confusion_matrix`. The grader counts torch calls for 64 pixels with 3 classes and for 19,200 pixels with 50 classes, and the two counts must be about the same.
- ❌ No floating-point counts in the accumulator.

---

<details>
  <summary>💡 Hint</summary>

  - Every metric here is a function of the confusion matrix. Which three vectors (one from the diagonal, two from sums) give TP, FP and FN for all classes at once?
  - How can a pair `(target, pred)` with both values in `[0, C)` become a **single** integer in `[0, C*C)`, so that a histogram of those integers *is* the flattened matrix? Which order makes the targets end up on the rows after `reshape(C, C)`?
  - What does `torch.tensor([200], dtype=torch.uint8) * 2` give?
  - What does `torch.tensor(0.0) / 0` give, and what does `torch.nanmean` do with it?
  - Counts add up across batches; metrics do not. What should the accumulator store?

</details>

In [ ]:
import torch

In [ ]:
# A hand-checkable example: C = 3 classes, 255 = void (unlabelled border pixels).
target = torch.tensor([[0, 0, 1, 1],
                       [0, 0, 1, 1],
                       [2, 2, 255, 255],
                       [2, 2, 255, 255]])
pred = torch.tensor([[0, 0, 1, 0],
                     [0, 1, 1, 1],
                     [2, 1, 2, 255],   # a prediction of 255 on a void pixel is never looked at
                     [2, 2, 1, 1]])
# Counted pixels: the 12 with target != 255.
#   target 0 -> predicted [0, 0, 0, 1]   row 0 = [3, 1, 0]
#   target 1 -> predicted [1, 0, 1, 1]   row 1 = [1, 3, 0]
#   target 2 -> predicted [2, 1, 2, 2]   row 2 = [0, 1, 3]
expected_cm = torch.tensor([[3, 1, 0], [1, 3, 0], [0, 1, 3]])


def cm_loop(pred, target, C, ignore_index=255):
    # Brute-force reference: one pixel at a time.
    cm = torch.zeros(C, C, dtype=torch.long)
    for p, t in zip(pred.flatten().tolist(), target.flatten().tolist()):
        if t != ignore_index and 0 <= t < C:
            cm[t, p] += 1
    return cm

In [ ]:
def confusion_matrix(pred: torch.Tensor, target: torch.Tensor, num_classes: int,
                     ignore_index=255) -> torch.Tensor:
    """
    Pixel confusion matrix of two integer label maps.

    Args:
        pred: integer tensor of predicted class ids, any shape.
        target: integer tensor of ground-truth class ids, same shape as pred.
        num_classes: C.
        ignore_index: target value to skip (None: skip nothing by value). Target values
            outside [0, C) are void and skipped too.

    Returns:
        (C, C) int64 tensor; cm[i, j] = number of counted pixels with target i and pred j.

    Raises:
        ValueError: if a counted pixel has a prediction outside [0, C).
    """
    # TODO: flatten; cast to int64 first
    # TODO: keep pixels whose target is in [0, C) and != ignore_index (if not None)
    # TODO: validate the kept predictions; raise ValueError if any is outside [0, C)
    # TODO: one bincount over a single integer code per (target, pred) pair -> reshape (C, C)
    raise NotImplementedError

In [ ]:
# Test 1: the confusion matrix
cm = confusion_matrix(pred, target, 3)
assert cm.dtype == torch.long and cm.shape == (3, 3), (cm.dtype, cm.shape)
assert torch.equal(cm, expected_cm), f"expected\n{expected_cm}\ngot\n{cm}"

# rows are ground truth: three class-0 pixels predicted as 1 -> cm[0, 1] == 3
assert confusion_matrix(torch.tensor([1, 1, 1]), torch.tensor([0, 0, 0]), 2).tolist() == [[0, 3], [0, 0]]

# random label maps of several shapes, with void pixels, against the pixel loop
g = torch.Generator().manual_seed(0)
for shape, C in [((50,), 2), ((3, 20, 30), 5), ((2, 2, 7, 9), 19)]:
    t = torch.randint(0, C, shape, generator=g)
    t[torch.rand(shape, generator=g) < 0.1] = 255
    p = torch.randint(0, C, shape, generator=g)
    assert torch.equal(confusion_matrix(p, t, C), cm_loop(p, t, C)), shape
    # uint8 masks (as loaded from PNG): 18 * 19 + 18 does not fit in a byte
    assert torch.equal(confusion_matrix(p.to(torch.uint8), t.to(torch.uint8), C), cm_loop(p, t, C)), "uint8"

# ignore_index variants
assert confusion_matrix(pred, target, 3, ignore_index=0)[0].sum() == 0, "ignore_index=0 drops target-0 pixels"
assert confusion_matrix(pred, target, 3, ignore_index=0)[:, 0].sum() == 1, "...but predictions of 0 still count"
assert confusion_matrix(pred, target, 3, ignore_index=None).sum() == 12, "255 is out of range -> void anyway"

# a bad prediction on a counted pixel is an error
try:
    confusion_matrix(torch.tensor([0, 3]), torch.tensor([0, 1]), 3)
    raise AssertionError("expected ValueError for a prediction of 3 with num_classes=3")
except ValueError:
    pass
print("Test 1 passed: confusion matrix")

In [ ]:
def iou_per_class(cm: torch.Tensor) -> torch.Tensor:
    """(C,) float64 IoU = TP / (TP + FP + FN); NaN for a class absent from target and prediction."""
    # TODO
    raise NotImplementedError


def mean_iou(cm: torch.Tensor) -> torch.Tensor:
    """0-dim float64: mean IoU over the classes present in target or prediction."""
    # TODO
    raise NotImplementedError


def dice_per_class(cm: torch.Tensor) -> torch.Tensor:
    """(C,) float64 Dice = 2TP / (2TP + FP + FN); NaN for an absent class."""
    # TODO
    raise NotImplementedError


def pixel_accuracy(cm: torch.Tensor) -> torch.Tensor:
    """0-dim float64: correctly labelled counted pixels / counted pixels."""
    # TODO
    raise NotImplementedError


def mean_class_accuracy(cm: torch.Tensor) -> torch.Tensor:
    """0-dim float64: mean over classes with ground-truth pixels of TP / (pixels of that class)."""
    # TODO
    raise NotImplementedError

In [ ]:
# Test 2: metrics on the hand example
# TP = [3, 3, 3], rows = [4, 4, 4], cols = [4, 5, 3] -> FN = [1, 1, 1], FP = [1, 2, 0]
iou = iou_per_class(expected_cm)
assert torch.allclose(iou, torch.tensor([3 / 5, 3 / 6, 3 / 4], dtype=torch.float64)), iou
assert abs(float(mean_iou(expected_cm)) - (0.6 + 0.5 + 0.75) / 3) < 1e-12
assert torch.allclose(dice_per_class(expected_cm), torch.tensor([6 / 8, 6 / 9, 6 / 7], dtype=torch.float64))
assert torch.allclose(dice_per_class(expected_cm), 2 * iou / (1 + iou)), "Dice = 2 IoU / (1 + IoU)"
assert abs(float(pixel_accuracy(expected_cm)) - 9 / 12) < 1e-12
assert abs(float(mean_class_accuracy(expected_cm)) - 0.75) < 1e-12

# a 4th class that never occurs and is never predicted: NaN, and the mean ignores it
cm4 = confusion_matrix(pred, target, 4)
assert torch.isnan(iou_per_class(cm4)[3]) and torch.isnan(dice_per_class(cm4)[3])
assert abs(float(mean_iou(cm4)) - float(mean_iou(expected_cm))) < 1e-12, "absent classes must not change mIoU"

# only predicted (class 1) / only in the ground truth (class 0) -> IoU 0, not NaN
only = torch.tensor([[0, 5], [0, 0]])
assert iou_per_class(only).tolist() == [0.0, 0.0]
assert abs(float(mean_class_accuracy(only)) - 0.0) < 1e-12   # class 1 has no GT pixels -> skipped
print("Test 2 passed: IoU, Dice, accuracies")

In [ ]:
class SegMetric:
    """Streaming dataset-level segmentation metrics: keeps only a (C, C) count matrix."""

    def __init__(self, num_classes: int, ignore_index=255):
        # TODO: store the config and an int64 (C, C) buffer
        raise NotImplementedError

    def reset(self):
        # TODO
        raise NotImplementedError

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        # TODO: add this batch's confusion matrix
        raise NotImplementedError

    def compute(self) -> dict:
        # TODO: keys confusion_matrix, iou, mean_iou, dice, mean_dice, pixel_accuracy,
        #       mean_class_accuracy -- all from the summed matrix; do not modify the state
        raise NotImplementedError

In [ ]:
# Test 3: dataset-level vs per-image mIoU
# Image A: 100 background pixels, except one pixel of class 1 that the model misses.
# Image B: 50 background + 50 class-1 pixels, all correct.
tA = torch.zeros(10, 10, dtype=torch.long); tA[0, 0] = 1
pA = torch.zeros(10, 10, dtype=torch.long)
tB = torch.zeros(10, 10, dtype=torch.long); tB[5:] = 1
pB = tB.clone()

metric = SegMetric(num_classes=2)
for p, t in ((pA, tA), (pB, tB)):
    metric.update(p, t)
res = metric.compute()
per_image = [float(mean_iou(confusion_matrix(p, t, 2))) for p, t in ((pA, tA), (pB, tB))]
print(f"per-image mIoU: {per_image} -> mean {sum(per_image) / 2:.4f}")
print(f"dataset-level mIoU: {float(res['mean_iou']):.4f}")
assert torch.equal(res["confusion_matrix"], torch.tensor([[149, 0], [1, 50]]))
assert abs(float(res["mean_iou"]) - (149 / 150 + 50 / 51) / 2) < 1e-12
assert set(res) >= {"confusion_matrix", "iou", "mean_iou", "dice", "mean_dice",
                    "pixel_accuracy", "mean_class_accuracy"}
assert torch.equal(metric.compute()["confusion_matrix"], res["confusion_matrix"]), "compute() must not change state"
metric.reset()
metric.update(pA, tA)
assert int(metric.compute()["confusion_matrix"].sum()) == 100
print("Test 3 passed: streaming, dataset-level")

In [ ]:
# Optional: cross-check the confusion matrix against sklearn when it is installed.
try:
    from sklearn.metrics import confusion_matrix as sk_confusion_matrix
except ImportError:
    print("sklearn not installed -- skipping the cross-check")
else:
    g = torch.Generator().manual_seed(1)
    t = torch.randint(0, 7, (4, 32, 32), generator=g)
    t[torch.rand(t.shape, generator=g) < 0.1] = 255
    p = torch.randint(0, 7, (4, 32, 32), generator=g)
    keep = t != 255
    ref = sk_confusion_matrix(t[keep].numpy(), p[keep].numpy(), labels=list(range(7)))
    assert (confusion_matrix(p, t, 7).numpy() == ref).all()
    print("matches sklearn.metrics.confusion_matrix (rows = true labels)")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("segmentation-metrics", confusion_matrix=confusion_matrix, iou_per_class=iou_per_class,
      mean_iou=mean_iou, dice_per_class=dice_per_class, pixel_accuracy=pixel_accuracy,
      mean_class_accuracy=mean_class_accuracy, SegMetric=SegMetric)